# v9_b Step 6: Port our network and KEGG to combined formula network



In [ ]:
import json
import pandas as pd

In [ ]:
outdir = 'results/step6_formula_networks/'

kegg_nodes = 'input/KEGG/map00140_cytoscape_nodes.csv'
kegg_edges = 'input/KEGG/map00140_cytoscape_edges.csv'

anno_dir = 'results/step5b_combined_net_anno/'
our_nodes = anno_dir + 'nodes_merged_metabolites.txt'
our_edges = anno_dir + 'edges_merged_metabolites.txt'

## Combined network with KEGG

In [ ]:
# KEGG: collapse compounds to formula
kn = pd.read_csv(kegg_nodes)
ke = pd.read_csv(kegg_edges)
cpd2formula = dict(zip(kn['id'], kn['formula']))

kegg_formula_nodes = kn.groupby('formula').agg(
    exact_mass=('exact_mass', 'first'),
    n_compounds=('id', 'size'),
    compound_ids=('id', ';'.join),
    compound_names=('name', lambda x: ';'.join(x.astype(str))),
).reset_index()

ke['source_formula'] = ke['source'].map(cpd2formula)
ke['target_formula'] = ke['target'].map(cpd2formula)
self_loops = ke['source_formula'] == ke['target_formula']   # isomerizations etc. collapse to self-loops
print('KEGG edges collapsing to self-loops (dropped):', self_loops.sum())

kegg_formula_edges = ke[~self_loops].groupby(['source_formula', 'target_formula']).agg(
    n_reactions=('reaction_id', 'nunique'),
    reaction_ids=('reaction_id', lambda x: ';'.join(sorted(set(x)))),
    reaction_names=('reaction_name', lambda x: ';'.join(sorted(set(x.dropna().astype(str))))),
    enzymes=('enzyme', lambda x: ';'.join(sorted(set(x.dropna().astype(str))))),
).reset_index().rename(columns={'source_formula': 'source', 'target_formula': 'target'})

print(len(kn), len(ke), '->', len(kegg_formula_nodes), len(kegg_formula_edges))

In [ ]:
# The edge file from step 5 has corrupted IDs (synonym str.replace hit prefixes, e.g. _product_3 in _product_38913),
# so edges are rebuilt from rxn_steps, which carry the formula of every step.
on = pd.read_csv(our_nodes, sep='\t')
features = json.load(open(anno_dir + 'merged_DnCl_features.json')
                     ) + json.load(open(anno_dir + 'merged_DnHz_features.json'))

edges, kept, dropped = {}, 0, 0
for feat in features:
    if 'M0,M+H+' in feat['ion_relation']:
        for rxn_steps in feat['rxn_steps']:
            # alternative annotations ending in another formula are skipped, to stay consistent with inferred_formula
            if rxn_steps[-1][1] != feat['inferred_formula']:
                dropped += 1
                continue
            kept += 1
            for ii in range(len(rxn_steps) - 1):
                key = (rxn_steps[ii][1], rxn_steps[ii + 1][1])
                e = edges.setdefault(key, {'rxn_formula_delta': set(), 'rxn_name': set(), 'n_paths': 0})
                e['rxn_formula_delta'].add(rxn_steps[ii][2])
                e['rxn_name'].add(rxn_steps[ii][3])
                e['n_paths'] += 1
print('rxn paths kept, dropped (alternative formula):', kept, dropped)

our_formula_edges = pd.DataFrame([
    {'source': s, 'target': t, 
     'rxn_formula_delta': ';'.join(sorted(v['rxn_formula_delta'])),
     'rxn_name': ' | '.join(sorted(v['rxn_name'])),
     'n_paths': v['n_paths']} for (s, t), v in edges.items() if s != t
])

def expand_sources(x):
    # 'DnCl_pellets;supernatant' -> DnCl_pellets, DnCl_supernatant
    out = set()
    for src in x:
        parts = src.split(';')
        out.add(parts[0])
        if len(parts) > 1:
            out.add(parts[0].split('_')[0] + '_' + parts[1])
    return ';'.join(sorted(out))

observed = on.groupby('inferred_formula').agg(
    n_features=('uid', 'size'),
    feature_uids=('uid', ';'.join),
    sources=('source', expand_sources),
)
all_formulas = sorted(set(our_formula_edges['source']) | set(our_formula_edges['target']) | set(on['inferred_formula']))
our_formula_nodes = pd.DataFrame({'formula': all_formulas}).merge(
    observed, left_on='formula', right_index=True, how='left')
our_formula_nodes['n_features'] = our_formula_nodes['n_features'].fillna(0).astype(int)
our_formula_nodes['observed'] = our_formula_nodes['n_features'] > 0     # False: predicted intermediate only
our_formula_nodes['is_substrate'] = our_formula_nodes['formula'].isin(on.loc[on['source'] == 'substrate', 'inferred_formula'])
our_formula_nodes['in_kegg'] = our_formula_nodes['formula'].isin(kegg_formula_nodes['formula'])
kegg_formula_nodes['in_ours'] = kegg_formula_nodes['formula'].isin(on['inferred_formula'])

print(len(on), '->', len(our_formula_nodes), 'formula nodes (observed:', our_formula_nodes['observed'].sum(), '),',
      len(our_formula_edges), 'formula edges')

In [ ]:
# Combined network: union of formula nodes and edges, labeled by presence (Our, KEGG, Both)
def presence(df):
    return df['_merge'].map({'left_only': 'Our', 'right_only': 'KEGG', 'both': 'Both'}).astype(str)

combined_formula_nodes = our_formula_nodes.drop(columns='in_kegg').merge(
    kegg_formula_nodes.drop(columns='in_ours'), on='formula', how='outer', indicator=True)
combined_formula_nodes['presence'] = presence(combined_formula_nodes)
combined_formula_nodes['observed'] = combined_formula_nodes['observed'].astype('boolean').fillna(False)
combined_formula_nodes['is_substrate'] = combined_formula_nodes['is_substrate'].astype('boolean').fillna(False)
combined_formula_nodes['n_features'] = combined_formula_nodes['n_features'].fillna(0).astype(int)
combined_formula_nodes['n_compounds'] = combined_formula_nodes['n_compounds'].fillna(0).astype(int)
combined_formula_nodes = combined_formula_nodes.drop(columns='_merge')

combined_formula_edges = our_formula_edges.merge(
    kegg_formula_edges, on=['source', 'target'], how='outer', indicator=True)
combined_formula_edges['presence'] = presence(combined_formula_edges)
combined_formula_edges['n_paths'] = combined_formula_edges['n_paths'].fillna(0).astype(int)
combined_formula_edges['n_reactions'] = combined_formula_edges['n_reactions'].fillna(0).astype(int)
combined_formula_edges = combined_formula_edges.drop(columns='_merge')

print(len(combined_formula_nodes), 'nodes', combined_formula_nodes['presence'].value_counts().to_dict())
print(len(combined_formula_edges), 'edges', combined_formula_edges['presence'].value_counts().to_dict())

In [ ]:
kegg_formula_nodes.to_csv(outdir + 'kegg_formula_nodes.tsv', sep='\t', index=False)
kegg_formula_edges.to_csv(outdir + 'kegg_formula_edges.tsv', sep='\t', index=False)
our_formula_nodes.to_csv(outdir + 'our_formula_nodes.tsv', sep='\t', index=False)
our_formula_edges.to_csv(outdir + 'our_formula_edges.tsv', sep='\t', index=False)
combined_formula_nodes.to_csv(outdir + 'combined_formula_nodes.tsv', sep='\t', index=False)
combined_formula_edges.to_csv(outdir + 'combined_formula_edges.tsv', sep='\t', index=False)

## Substrate-specific networks

Product-level networks for 5 substrates: nodes are predicted product IDs (e.g. `estradiol_product_140`), edges are reaction steps.
Each substrate has a subnetwork of up to 2 reaction steps and the full network of up to 3 steps.

The substrate-specific products have unique formulas, so no need to draw formula level network on each substrate.
The only exception is a 2-step product with the same formula as its substrate (`pregnenolone_product_170`, `progesterone_product_170`).

In [ ]:
# Each rxn_steps path starts at its substrate, so paths are split by rxn_steps[0][0];
# same filters as above: M+H+ features only, paths must end in the feature's inferred_formula.
substrates = {
    'merged_DnCl_features.json': ['Estrone', 'Estradiol'],
    'merged_DnHz_features.json': ['DHEA', 'Pregnenolone', 'Progesterone'],
}

def substrate_network(substrate, features, max_steps):
    nodes, edges = {}, {}
    for feat in features:
        if 'M0,M+H+' not in feat['ion_relation']:
            continue
        for rxn_steps in feat['rxn_steps']:
            if (rxn_steps[0][0] != substrate or rxn_steps[-1][1] != feat['inferred_formula']
                    or len(rxn_steps) - 1 > max_steps):
                continue
            for ii, step in enumerate(rxn_steps):
                nodes.setdefault(step[0], {'formula': step[1], 'step': ii,
                                           'feature_ids': set(), 'sources': set(), 'enzymes': set()})
            o = nodes[rxn_steps[-1][0]]     # the feature is matched to the last product of the path
            o['feature_ids'].add(feat['id'])
            o['sources'].add(feat['source'])
            enzyme_groups = []
            if isinstance(feat['bioreactions_pellets'], list):
                enzyme_groups.append(feat['bioreactions_pellets'])
            if isinstance(feat['bioreactions_supernatant'], list):
                enzyme_groups.extend(feat['bioreactions_supernatant'])
            for group in enzyme_groups:
                o['enzymes'].update(enz for enz, sub in group if sub == substrate.lower())
            for ii in range(len(rxn_steps) - 1):
                key = (rxn_steps[ii][0], rxn_steps[ii + 1][0])
                e = edges.setdefault(key, {'rxn_formula_delta': rxn_steps[ii][2], 'rxn_name': rxn_steps[ii][3], 'n_paths': 0})
                e['n_paths'] += 1

    ndf = pd.DataFrame([
        {'id': k, 'formula': v['formula'], 'step': v['step'],
         'n_features': len(v['feature_ids']),
         'feature_ids': ';'.join(sorted(v['feature_ids'])),
         'sources': expand_sources(v['sources']),
         'enzymes': ';'.join(sorted(v['enzymes'])),
         } for k, v in nodes.items()
    ]).sort_values(['step', 'id'])
    ndf['observed'] = ndf['n_features'] > 0     # False: predicted intermediate only
    ndf['is_substrate'] = ndf['step'] == 0
    ndf['in_kegg'] = ndf['formula'].isin(kegg_formula_nodes['formula'])
    edf = pd.DataFrame([{'source': s, 'target': t, **v} for (s, t), v in edges.items()])
    return ndf, edf

In [ ]:
for json_file, substrate_list in substrates.items():
    features = json.load(open(anno_dir + json_file))
    for substrate in substrate_list:
        for max_steps in [2, 3]:
            ndf, edf = substrate_network(substrate, features, max_steps)
            prefix = outdir + f'{substrate.lower()}_{max_steps}step_products_'
            ndf.to_csv(prefix + 'nodes.tsv', sep='\t', index=False)
            edf.to_csv(prefix + 'edges.tsv', sep='\t', index=False)
            print(substrate, f'up to {max_steps} steps:', len(ndf), 'nodes (observed:', ndf['observed'].sum(),
                  ', formula in KEGG:', ndf['in_kegg'].sum(), '),', len(edf), 'edges')

# v9_b Step 6: Summary

Network files are saved for CytoScape visualization. 

The "combined_formula_*.tsv" files contain all ours and KEGG pathway. 